En este notebook vamos a hacer una simulación del mes de febrero con nuestra estimación de estaciones (y sus respectivos tamaños).

In [2]:
import pandas as pd

estaciones = pd.read_csv('centroides2.csv')
viajes = pd.read_csv('potenciales_viajes_cubiertos.csv')
tamaño_estaciones = pd.read_csv('tamaño_cluster (1).csv')

viajes['tsO'] = pd.to_datetime(viajes['tsO'])
viajes['tsD'] = pd.to_datetime(viajes['tsD'])

In [ ]:
viajes.head()

,Unnamed: 0,idS,tsO,tsD,price,tt,dis,vel,lonO,latO,lonD,latD,duration_calc,clusterOrigen,clusterFinal,origin_lat,origin_lon,destination_lat,destination_lon
0,0,A0H4,2021-02-03 18:10:03,2021-02-03 18:17:44,2.1525,461,1715.336751,13.395254,12.466222,41.867388,12.470660,41.853908,461.0,174.0,122.0,NaN,NaN,NaN,NaN
1,1,A0H4,2021-02-13 18:21:13,2021-02-13 18:25:33,1.6500,260,1234.472044,17.092690,12.471143,41.923692,12.467502,41.934306,260.0,59.0,98.0,NaN,NaN,NaN,NaN
2,2,A0H4,2021-02-14 13:39:54,2021-02-14 13:48:03,2.2225,489,2221.481536,16.354465,12.467524,41.934342,12.486330,41.928270,489.0,98.0,137.0,NaN,NaN,NaN,NaN
3,3,A0H4,2021-02-14 14:37:53,2021-02-14 14:57:53,4.0000,1200,4562.843566,13.688531,12.486275,41.928301,12.457922,41.904302,1200.0,137.0,113.0,NaN,NaN,NaN,NaN
4,4,A0H4,2021-02-15 13:31:24,2021-02-15 13:34:45,1.5025,201,550.154792,9.853519,12.457876,41.904303,12.460773,41.907606,201.0,113.0,179.0,NaN,NaN,NaN,NaN


In [61]:
import pandas as pd
import numpy as np

def calcular_optimos_por_periodos(viajes_mes, estaciones_df, fechas_redistribucion):
    """
    Calcula el estado inicial óptimo para el INICIO de cada periodo de redistribución.
    El cálculo considera la acumulación de demanda durante todo el periodo (ej. una semana).
    """

    # 1. Preparar Flujos (Igual que antes)
    salidas = viajes_mes[['tsO', 'clusterOrigen']].copy()
    salidas.columns = ['time', 'station_id']
    salidas['change'] = -1

    llegadas = viajes_mes[['tsD', 'clusterFinal']].copy()
    llegadas.columns = ['time', 'station_id']
    llegadas['change'] = 1

    flujo_total = pd.concat([salidas, llegadas]).sort_values('time')

    # 2. ASIGNAR PERIODOS
    # Convertimos las fechas de corte a datetime para comparar
    fechas_corte = pd.to_datetime(fechas_redistribucion)

    # Usamos searchsorted para ver en qué 'cajón' (periodo) cae cada viaje
    # Esto asigna 0 al primer intervalo, 1 al segundo, etc.
    # Nota: Asegúrate de que 'time' esté ordenado para searchsorted, o usa apply (más lento)
    # Aquí usamos un método vectorial robusto:
    flujo_total['period_start_date'] = pd.cut(
        flujo_total['time'],
        bins=list(fechas_corte) + [pd.Timestamp.max],
        labels=fechas_redistribucion,
        right=False
    )

    # Eliminamos viajes que queden fuera de los rangos (si los hay)
    flujo_total = flujo_total.dropna(subset=['period_start_date'])

    optimos_por_periodo = {} # Key: Fecha de inicio del periodo, Value: Dict de estaciones

    # 3. Iteramos por PERIODO (ej: Semana 1, Semana 2...)
    grupos_periodo = flujo_total.groupby('period_start_date')

    print(f"Calculando optimización para {len(grupos_periodo)} periodos de redistribución...")

    for fecha_inicio, flujo_periodo in grupos_periodo:

        optimum_states = {}
        grupos_estacion = flujo_periodo.groupby('station_id')

        for _, row in estaciones_df.iterrows():
            st_id = int(row['Station'])
            col_name = f'station{st_id}'

            df_st = grupos_estacion.get_group(st_id).sort_values('time')

            # --- LA CLAVE ---
            # El cumsum ahora recorre VARIOS DÍAS.
            # Si el lunes pierdes 2 bicis y el martes pierdes 3, el acumulado llega a -5.
            cumsum = df_st['change'].cumsum()

            min_reach = cumsum.min()
            max_reach = cumsum.max()

            # Lógica de optimización idéntica, pero aplicada a la curva semanal
            needed_start = abs(min_reach) if min_reach < 0 else 0

            peak_occupancy = needed_start + max_reach
            final_start = needed_start

            # El conflicto es MUCHO más probable en periodos largos
            if peak_occupancy > 16:
                mid_flow = (max_reach + min_reach) / 2
                final_start = (16 / 2) - mid_flow

            final_start = max(0, min(16, np.round(final_start)))
            optimum_states[col_name] = (peak_occupancy if peak_occupancy <= 16 else 16, final_start)

        optimos_por_periodo[fecha_inicio] = optimum_states

    return optimos_por_periodo

# --- CONFIGURACIÓN ---
# Definimos los lunes de febrero como días de redistribución
dias_redistribucion = ['2021-02-01', '2021-02-08', '2021-02-15', '2021-02-22']

# Calculamos
optimos_semanales = calcular_optimos_por_periodos(viajes, tamaño_estaciones, dias_redistribucion)

/tmp/ipython-input-3703383016.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grupos_periodo = flujo_total.groupby('period_start_date')


Calculando optimización para 4 periodos de redistribución...


In [63]:
import numpy as np
import pandas as pd


# --- 2. CÁLCULO DE LA CAPACIDAD MÁXIMA MENSUAL POR ESTACIÓN ---
max_capacities = {}
for semana, estados in optimos_semanales.items():
    for id_estacion, (capacidad, _) in estados.items():
        # Convierte el valor de capacidad a un tipo estándar para la comparación (int)
        capacidad_val = int(capacidad) if isinstance(capacidad, (np.int64, np.float64)) else capacidad

        # Si la estación no está en el diccionario o la capacidad actual es mayor que la guardada,
        # actualizamos el valor máximo.
        if id_estacion not in max_capacities or capacidad_val > max_capacities[id_estacion]:
            max_capacities[id_estacion] = capacidad_val

# --- 3. MODIFICACIÓN DEL DICCIONARIO optimos_semanales ---
optimos_semanales_modificado = {}

for semana, estados in optimos_semanales.items():
    nuevos_estados = {}
    for id_estacion, (_, ocupacion_inicial) in estados.items():
        # Obtenemos la capacidad máxima calculada en el paso 2
        max_capacidad = max_capacities[id_estacion]

        # Aseguramos que la ocupación inicial no exceda la nueva capacidad máxima
        # (Aunque no es el objetivo principal, es una buena práctica de seguridad)
        ocupacion_final = min(max_capacidad, ocupacion_inicial)

        # El nuevo estado para esta estación y semana es (Capacidad_Máxima_Mensual, Ocupación_Inicial_Original)
        nuevos_estados[id_estacion] = (max_capacidad, ocupacion_final)

    optimos_semanales_modificado[semana] = nuevos_estados

# Asignar el diccionario modificado a la variable original
optimos_semanales = optimos_semanales_modificado

# --- VERIFICACIÓN (Opcional) ---
print("✅ Proceso completado. El diccionario 'optimos_semanales' ha sido modificado.")
print("\nVerificación de estaciones (Capacidad original vs. Capacidad máxima):")

# Muestra el estado de la estación0 en todas las semanas:
for semana, estados in optimos_semanales.items():
    print(f"[{semana}]: station0 -> {estados['station0']}")

# Muestra el estado de la estación2 en todas las semanas:
for semana, estados in optimos_semanales.items():
    print(f"[{semana}]: station2 -> {estados['station2']}")

✅ Proceso completado. El diccionario 'optimos_semanales' ha sido modificado.

Verificación de estaciones (Capacidad original vs. Capacidad máxima):
[2021-02-01]: station0 -> (16, 0)
[2021-02-08]: station0 -> (16, np.int64(3))
[2021-02-15]: station0 -> (16, np.int64(3))
[2021-02-22]: station0 -> (16, np.int64(1))
[2021-02-01]: station2 -> (12, np.int64(7))
[2021-02-08]: station2 -> (12, 12)
[2021-02-15]: station2 -> (12, np.int64(10))
[2021-02-22]: station2 -> (12, np.int64(4))


In [65]:
import numpy as np
import pandas as pd

# Definir los tiempos de inicio y fin
start_time = f'2021-02-01 00:00:00'
end_time = f'2021-02-28 23:59:00'
minutos = pd.date_range(start=start_time, end=end_time, freq='min')

# Crear un DataFrame vacío para las simulaciones con las estaciones como columnas
station_columns = [f'station{i}' for i in range(190)]  # Asegurarse de tener 190 estaciones
simulations_df = pd.DataFrame(index=minutos, columns=station_columns)

# --- Definición de los días de inicio de semana ---
# Estos días son donde aplicaremos el estado 'óptimo'
dias_inicio_semana = {
    '2021-02-01',  # Lunes de la primera semana
    '2021-02-08',  # Lunes de la segunda semana
    '2021-02-15',  # Lunes de la tercera semana
    '2021-02-22'   # Lunes de la cuarta semana
}

# La función para obtener el estado ya no necesita lógica de rango,
# solo verifica si la fecha es un día de inicio de semana y si está en optimos_semanales
def obtener_estado_inicial(fecha):
    # Aquí se asume que 'optimos_semanales' está disponible y es un diccionario
    # cuya clave es la fecha de inicio de la semana y el valor son los estados (capacidad, ocupación)

    # Solo devolvemos un estado si la fecha coincide exactamente con un día de inicio de semana
    if fecha in dias_inicio_semana:
        # Nota: La implementación asume que el diccionario ya está filtrado por estas claves
        return optimos_semanales[fecha]
    else:
        # Devolvemos None o un valor que indique que no hay estado inicial para aplicar
        return None

# Establecer el estado inicial en el primer minuto (2021-02-01 00:00:00)
fecha_inicio = minutos[0].strftime('%Y-%m-%d')
estado_inicial = obtener_estado_inicial(fecha_inicio)
# Solo aplicamos si hay un estado para la fecha, lo cual será el caso para 2021-02-01
if estado_inicial is not None:
    simulations_df.loc[minutos[0]] = [estado_inicial[col] for col in station_columns]
    # Usamos una lista de comprension para asignar las tuplas directamente
else:
    # Manejo de error si el primer día no tiene estado inicial definido
    print(f"Error: No se encontró estado inicial para el primer día {fecha_inicio}.")


# Crear listas para manejar los errores y los índices a saltar
errores_df = []
saltar = []

# Asumimos que tienes el dataframe 'viajes' con la columna 'tsO' para los viajes de salida
# y 'tsD' para los viajes de llegada

for i, minuto in enumerate(minutos[1:]):
    # Paso 1: Copiar el estado del minuto anterior
    # Esta es la base para la simulación de flujo libre
    simulations_df.loc[minuto] = simulations_df.loc[minutos[i]]

    # Paso 2: Aplicar el Estado Inicial SOLO en el primer minuto de los días de inicio de semana
    fecha_actual = minuto.strftime('%Y-%m-%d')
    hora_actual = minuto.strftime('%H:%M:%S')

    # Verificamos si es un día de inicio de semana Y el primer minuto del día
    if fecha_actual in dias_inicio_semana and hora_actual == '00:00:00':
        estado_optimo_semanal = obtener_estado_inicial(fecha_actual)

        if estado_optimo_semanal is not None:
            # Sobreescribir el estado para el minuto 00:00:00 con el óptimo
            for station_col in station_columns:
                if station_col in estado_optimo_semanal:
                    # Aplicamos la capacidad y ocupación del óptimo semanal
                    simulations_df.loc[minuto, station_col] = estado_optimo_semanal[station_col]
                else:
                    # Manejar estaciones faltantes si es necesario
                    pass

    # --- A partir de aquí, la simulación evoluciona por los viajes ---

    # Trips starting at this minute (Viajes de Salida)
    # Se utiliza el estado de 'simulations_df.loc[minuto]' que ya ha sido inicializado/copiado
    trips_starting_now = viajes[viajes['tsO'].dt.floor('min') == minuto]
    for index, row in trips_starting_now.iterrows():
        origen_cluster = int(row['clusterOrigen'])
        station_col = f'station{origen_cluster}'

        if station_col in simulations_df.columns:
            # Asegurarse de que el valor es una tupla (capacidad, ocupación)
            try:
                current_capacity, current_occupation = simulations_df.loc[minuto, station_col]
            except ValueError:
                 # Esto puede ocurrir si el valor aún no es una tupla, intentar recuperarlo del minuto anterior si es necesario
                 current_capacity, current_occupation = simulations_df.loc[minutos[i], station_col]

            if current_occupation == 0:
                # Error: No hay bicicletas disponibles
                errores_df.append({'Error': 'Falta bicis', 'Station': station_col, 'Minuto': minuto})
                saltar.append(index)
                # No se pudo tomar la bici, el estado permanece igual, pero aseguramos que la ocupación sea 0.0
                simulations_df.loc[minuto, station_col] = (current_capacity, 0.0)
            else:
                # Éxito: Se toma una bici
                simulations_df.loc[minuto, station_col] = (current_capacity, max(0.0, current_occupation - 1.0))

    # Trips ending at this minute (Viajes de Llegada)
    trips_ending_now = viajes[viajes['tsD'].dt.floor('min') == minuto]
    for index, row in trips_ending_now.iterrows():
        # Si el viaje fue saltado por falta de bici en origen, se ignora la llegada
        if index in saltar:
            continue
        else:
            destino_cluster = int(row['clusterFinal'])
            station_col = f'station{destino_cluster}'

            if station_col in simulations_df.columns:
                 # Asegurarse de que el valor es una tupla (capacidad, ocupación)
                try:
                    current_capacity, current_occupation = simulations_df.loc[minuto, station_col]
                except ValueError:
                    # Intentar recuperarlo del estado que ya se modificó por las salidas en este mismo minuto
                    current_capacity, current_occupation = simulations_df.loc[minuto, station_col]

                if current_occupation == current_capacity:
                    # Error: La estación está llena
                    errores_df.append({'Error': 'Falta espacio', 'Station': station_col, 'Minuto': minuto})
                    # No se puede aparcar, el estado permanece igual (ocupación = capacidad máxima)
                    simulations_df.loc[minuto, station_col] = (current_capacity, current_capacity)
                else:
                    # Éxito: Se aparca una bici
                    simulations_df.loc[minuto, station_col] = (current_capacity, min(current_capacity, current_occupation + 1.0))

# Convertir los errores en un DataFrame
errores_df = pd.DataFrame(errores_df)

print("Simulation completed for the entire month with weekly rebalancing at 00:00:00.")
errores_df

Simulation completed for the entire month with weekly rebalancing at 00:00:00.


,Error,Station,Minuto
0,Falta bicis,station133,2021-02-01 07:49:00
1,Falta espacio,station61,2021-02-01 07:52:00
2,Falta bicis,station133,2021-02-01 08:10:00
3,Falta bicis,station0,2021-02-01 08:31:00
4,Falta bicis,station133,2021-02-01 08:33:00
...,...,...,...
480,Falta bicis,station61,2021-02-28 20:38:00
481,Falta bicis,station55,2021-02-28 20:54:00
482,Falta bicis,station50,2021-02-28 21:29:00
483,Falta espacio,station133,2021-02-28 21:53:00


In [66]:
simulations_df

,station0,station1,station2,station3,station4,station5,station6,station7,station8,station9,...,station180,station181,station182,station183,station184,station185,station186,station187,station188,station189
2021-02-01 00:00:00,"(16, 0)","(16, 8)","(12, 7)","(11, 6)","(11, 3)","(15, 7)","(16, 4)","(7, 4)","(12, 0)","(9, 1)",...,"(8, 1)","(8, 8)","(8, 0)","(8, 8)","(16, 5)","(16, 0)","(11, 8)","(6, 2)","(7, 2)","(16, 0)"
2021-02-01 00:01:00,"(16, 0)","(16, 8)","(12, 7)","(11, 6)","(11, 3)","(15, 7)","(16, 4)","(7, 4)","(12, 0)","(9, 1)",...,"(8, 1)","(8, 8)","(8, 0)","(8, 8)","(16, 5)","(16, 0)","(11, 8)","(6, 2)","(7, 2)","(16, 0)"
2021-02-01 00:02:00,"(16, 0)","(16, 8)","(12, 7)","(11, 6)","(11, 3)","(15, 7)","(16, 4)","(7, 4)","(12, 0)","(9, 1)",...,"(8, 1)","(8, 8)","(8, 0)","(8, 8)","(16, 5)","(16, 0)","(11, 8)","(6, 2)","(7, 2)","(16, 0)"
2021-02-01 00:03:00,"(16, 0)","(16, 8)","(12, 7)","(11, 6)","(11, 3)","(15, 7)","(16, 4)","(7, 4)","(12, 0)","(9, 1)",...,"(8, 1)","(8, 8)","(8, 0)","(8, 8)","(16, 5)","(16, 0)","(11, 8)","(6, 2)","(7, 2)","(16, 0)"
2021-02-01 00:04:00,"(16, 0)","(16, 8)","(12, 7)","(11, 6)","(11, 3)","(15, 7)","(16, 4)","(7, 4)","(12, 0)","(9, 1)",...,"(8, 1)","(8, 8)","(8, 0)","(8, 8)","(16, 5)","(16, 0)","(11, 8)","(6, 2)","(7, 2)","(16, 0)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-02-28 23:55:00,"(16, 11.0)","(16, 0.0)","(12, 4.0)","(11, 10.0)","(11, 4.0)","(15, 8.0)","(16, 15.0)","(7, 2.0)","(12, 0.0)","(9, 5.0)",...,"(8, 2.0)","(8, 1.0)","(8, 3.0)","(8, 1.0)","(16, 0.0)","(16, 4.0)","(11, 7.0)","(6, 2.0)","(7, 4.0)","(16, 1.0)"
2021-02-28 23:56:00,"(16, 11.0)","(16, 0.0)","(12, 4.0)","(11, 10.0)","(11, 4.0)","(15, 8.0)","(16, 15.0)","(7, 2.0)","(12, 0.0)","(9, 5.0)",...,"(8, 2.0)","(8, 1.0)","(8, 3.0)","(8, 1.0)","(16, 0.0)","(16, 4.0)","(11, 7.0)","(6, 2.0)","(7, 4.0)","(16, 1.0)"
2021-02-28 23:57:00,"(16, 11.0)","(16, 0.0)","(12, 4.0)","(11, 10.0)","(11, 4.0)","(15, 8.0)","(16, 15.0)","(7, 2.0)","(12, 0.0)","(9, 5.0)",...,"(8, 2.0)","(8, 1.0)","(8, 3.0)","(8, 1.0)","(16, 0.0)","(16, 4.0)","(11, 7.0)","(6, 2.0)","(7, 4.0)","(16, 1.0)"
2021-02-28 23:58:00,"(16, 11.0)","(16, 0.0)","(12, 4.0)","(11, 10.0)","(11, 4.0)","(15, 8.0)","(16, 15.0)","(7, 2.0)","(12, 0.0)","(9, 5.0)",...,"(8, 2.0)","(8, 1.0)","(8, 3.0)","(8, 1.0)","(16, 0.0)","(16, 4.0)","(11, 7.0)","(6, 2.0)","(7, 4.0)","(16, 1.0)"


In [16]:
simulations_df.loc[optimos_semanales.keys()]

,station0,station1,station2,station3,station4,station5,station6,station7,station8,station9,...,station180,station181,station182,station183,station184,station185,station186,station187,station188,station189
2021-02-01,"(10.0, 0)","(11.0, 8)","(8.0, 7)","(8.0, 6)","(9.0, 3)","(8.0, 6.0)","(9.0, 2.0)","(4.0, 3.0)","(6.0, 0)","(6.0, 0)",...,"(7.0, 1)","(6.0, 6.0)","(5.0, 0)","(6.0, 6.0)","(8.0, 5)","(7.0, 0)","(7.0, 7.0)","(4.0, 2)","(5.0, 2)","(7.0, 0)"
2021-02-08,"(10.0, 3)","(11.0, 11.0)","(8.0, 8.0)","(8.0, 8.0)","(9.0, 9.0)","(8.0, 1)","(9.0, 0)","(4.0, 0)","(6.0, 6.0)","(6.0, 1.0)",...,"(7.0, 0)","(6.0, 6.0)","(5.0, 1)","(6.0, 6.0)","(8.0, 8.0)","(7.0, 0)","(7.0, 2)","(4.0, 0)","(5.0, 4)","(7.0, 0)"
2021-02-15,"(10.0, 2.0)","(11.0, 6.0)","(8.0, 8.0)","(8.0, 6)","(9.0, 5)","(8.0, 0)","(9.0, 1.0)","(4.0, 0)","(6.0, 6.0)","(6.0, 0)",...,"(7.0, 2)","(6.0, 4.0)","(5.0, 1.0)","(6.0, 3)","(8.0, 0)","(7.0, 3.0)","(7.0, 1.0)","(4.0, 0)","(5.0, 2.0)","(7.0, 0)"
2021-02-22,"(10.0, 0)","(11.0, 10.0)","(8.0, 4)","(8.0, 0)","(9.0, 6)","(8.0, 0)","(9.0, 0)","(4.0, 1)","(6.0, 6.0)","(6.0, 0)",...,"(7.0, 0)","(6.0, 6.0)","(5.0, 1.0)","(6.0, 0)","(8.0, 8.0)","(7.0, 0)","(7.0, 5)","(4.0, 0)","(5.0, 1)","(7.0, 0)"


In [67]:
count = {}
for semana in optimos_semanales.keys():
  for keys in optimos_semanales[semana].keys():
    if semana not in count:
      count[semana] = optimos_semanales[semana][keys][1]
    else:
      count[semana] += optimos_semanales[semana][keys][1]
print(count)

{'2021-02-01': np.float64(789.0), '2021-02-08': np.float64(769.0), '2021-02-15': np.float64(829.0), '2021-02-22': np.float64(841.0)}
